In [1]:
import pandas as pd
df = pd.read_csv("titanic.csv")
df.shape

(891, 12)

In [2]:
pd.cut(df["Age"], bins=[0, 18, 40, 60, 100])

0      (18.0, 40.0]
1      (18.0, 40.0]
2      (18.0, 40.0]
3      (18.0, 40.0]
4      (18.0, 40.0]
           ...     
886    (18.0, 40.0]
887    (18.0, 40.0]
888             NaN
889    (18.0, 40.0]
890    (18.0, 40.0]
Name: Age, Length: 891, dtype: category
Categories (4, interval[int64, right]): [(0, 18] < (18, 40] < (40, 60] < (60, 100]]

In [3]:
df["AgeBin"] = pd.cut(df["Age"],
                      bins=[0, 18, 40, 60, 100],
                      labels=["Child", "Young Adult", "Middle Aged", "Senior"])
df["AgeBin"].value_counts()

AgeBin
Young Adult    425
Child          139
Middle Aged    128
Senior          22
Name: count, dtype: int64

In [4]:
df.groupby("AgeBin", observed=True)["Survived"].mean()

AgeBin
Child          0.503597
Young Adult    0.388235
Middle Aged    0.390625
Senior         0.227273
Name: Survived, dtype: float64

In [5]:
pd.cut(df["Age"], bins=[0, 18, 40, 60, 100], right=False).value_counts()

Age
[18, 40)     438
[40, 60)     137
[0, 18)      113
[60, 100)     26
Name: count, dtype: int64

In [6]:
pd.qcut(df["Fare"], q=4, labels=["Low", "Mid", "High", "Very High"]).value_counts()


Fare
Mid          224
Low          223
High         222
Very High    222
Name: count, dtype: int64

In [7]:
pd.qcut(df["Fare"], q=4, labels=["Low", "Mid", "High", "Very High"]).value_counts()

Fare
Mid          224
Low          223
High         222
Very High    222
Name: count, dtype: int64

In [8]:
# Bin Fare into four equal-width ranges using cut with your own edges. Compare the group sizes to the qcut result (224/223/222/222). Explain what the difference tells you about how fares are distributed.

print(pd.cut(df["Fare"], bins=[0, 25, 50, 75, 100]).value_counts())

# Inside qcut the values for given labels are very much closer to each other and in cut the edges we defined had values very varying with each other as fare were concentrated heavily on lower values

Fare
(0, 25]      542
(25, 50]     174
(50, 75]      63
(75, 100]     44
Name: count, dtype: int64


In [9]:
#Bin Age into ranges with right=False and assign it as a column. Get survival rate by that column and compare to the right=True version. Did the 26 boundary passengers change any conclusion?

bins = [0, 13, 26, 40, 80]

df["AgeGroup_left"] = pd.cut(
    df["Age"],
    bins=bins,
    right=False
)

print(df.groupby("AgeGroup_left", observed=False)["Survived"].mean())

df["AgeGroup_right"] = pd.cut(df["Age"], bins=bins, right=True)

print(df.groupby("AgeGroup_right", observed=False)["Survived"].mean())

# values changes slightly


AgeGroup_left
[0, 13)     0.579710
[13, 26)    0.362069
[26, 40)    0.420000
[40, 80)    0.370370
Name: Survived, dtype: float64
AgeGroup_right
(0, 13]     0.591549
(13, 26]    0.354839
(26, 40]    0.428571
(40, 80]    0.366667
Name: Survived, dtype: float64


In [10]:
# Use qcut on Age with q=3. Report the edges pandas chose. Would you use cut or qcut for age brackets in a report, and why?
print(pd.qcut(df["Age"], q=3).cat.categories)

IntervalIndex([(0.419, 23.0], (23.0, 34.0], (34.0, 80.0]], dtype='interval[float64, right]')


In [11]:
pd.crosstab(df["Pclass"], df["Sex"])

Sex,female,male
Pclass,,
1,94,122
2,76,108
3,144,347


In [12]:
pd.crosstab(df["Pclass"], df["Sex"], normalize="index")

Sex,female,male
Pclass,,
1,0.435185,0.564815
2,0.413043,0.586957
3,0.293279,0.706721


In [13]:
pd.crosstab(df["Pclass"], df["Sex"], normalize="columns")

Sex,female,male
Pclass,,
1,0.299363,0.211438
2,0.242038,0.187175
3,0.458599,0.601386


In [14]:
pd.crosstab(df["Pclass"], df["Sex"], values=df["Survived"], aggfunc="mean")

Sex,female,male
Pclass,,
1,0.968085,0.368852
2,0.921053,0.157407
3,0.500000,0.135447


In [15]:
pd.crosstab(df["AgeBin"], df["Survived"], normalize="index")

Survived,0,1
AgeBin,,
Child,0.496403,0.503597
Young Adult,0.611765,0.388235
Middle Aged,0.609375,0.390625
Senior,0.772727,0.227273


In [16]:
# Crosstab Embarked against Pclass. Which port sent the most third-class passengers?
pd.crosstab(df["Embarked"], df["Pclass"])
# S

Pclass,1,2,3
Embarked,,,
C,85,17,66
Q,2,3,72
S,127,164,353


In [17]:
# Same table with normalize="index", then normalize="columns". Write one sentence for each stating what the denominator is. Both sentences must start "Of all…
pd.crosstab(df["Embarked"], df["Pclass"], normalize="index")


Pclass,1,2,3
Embarked,,,
C,0.505952,0.101190,0.392857
Q,0.025974,0.038961,0.935065
S,0.197205,0.254658,0.548137


In [18]:
pd.crosstab(df["Embarked"], df["Pclass"], normalize="columns")

# Of all passengers from each port, this shows the proportion belonging to each passenger class.
# Of all passengers in each passenger class, this shows the proportion who came from each port.

Pclass,1,2,3
Embarked,,,
C,0.397196,0.092391,0.134420
Q,0.009346,0.016304,0.146640
S,0.593458,0.891304,0.718941


In [19]:
survival_age_sex = pd.crosstab(
    df["AgeBin"],
    df["Sex"],
    values=df["Survived"],
    aggfunc="mean"
)

print(survival_age_sex)

print("Highest:")
print(survival_age_sex.stack().idxmax(), survival_age_sex.stack().max())

print("Lowest:")
print(survival_age_sex.stack().idxmin(), survival_age_sex.stack().min())

Sex            female      male
AgeBin                         
Child        0.676471  0.338028
Young Adult  0.786207  0.182143
Middle Aged  0.755556  0.192771
Senior       1.000000  0.105263
Highest:
('Senior', 'female') 1.0
Lowest:
('Senior', 'male') 0.10526315789473684


In [20]:
pd.crosstab(
    df["Embarked"],
    df["Pclass"],
    margins=True
)

Pclass,1,2,3,All
Embarked,,,,
C,85,17,66,168
Q,2,3,72,77
S,127,164,353,644
All,214,184,491,889


In [22]:
df["FareBin"] = pd.qcut(df["Fare"], q=4, labels=["Low", "Mid", "High", "Very High"])

pd.crosstab(
    df["FareBin"],
    df["Survived"],
    normalize="index"
)

Survived,0,1
FareBin,,
Low,0.802691,0.197309
Mid,0.696429,0.303571
High,0.545045,0.454955
Very High,0.418919,0.581081


In [23]:
pd.crosstab(df["FareBin"], df["Pclass"])

Pclass,1,2,3
FareBin,,,
Low,6,6,211
Mid,0,86,138
High,51,70,101
Very High,159,22,41


In [24]:
pd.crosstab(df["Pclass"], df["FareBin"], values=df["Survived"], aggfunc="mean")

FareBin,Low,Mid,High,Very High
Pclass,,,,
1,0.000000,NaN,0.529412,0.685535
2,0.000000,0.383721,0.600000,0.545455
3,0.208531,0.253623,0.316832,0.195122


In [ ]:
# Conclusion: Fare still contains some information within 3rd class, but the relationship is not simply “higher fare = higher survival.”